# QuTiP introduction

Cleaned and organized for reproducible execution from the repository root.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image
from qutip import (Qobj, about, basis, coherent, coherent_dm, create, destroy,
                   expect, fock, fock_dm, mesolve, qeye, sigmax, sigmay,
                   sigmaz, tensor, thermal_dm, anim_matrix_histogram,
                   anim_fock_distribution, fock, plot_fock_distribution, wigner, squeeze, displace,ket2dm,num)
# set a parameter to see animations in line
from matplotlib import rc
rc('animation', html='jshtml')
import matplotlib.animation as animation
from IPython.display import HTML
%matplotlib inline

## Creating and inspecting quantum objects

In [ ]:
q = Qobj([[1], [0]])

q

In [ ]:
# the dimension, or composite Hilbert state space structure
q.dims

In [ ]:
# the shape of the matrix data representation
q.shape

In [ ]:
# the matrix data itself. in sparse matrix format.
q.data

In [ ]:
# get the dense matrix representation
q.full()

In [ ]:
# some additional properties
q.isherm, q.type

## Using Qobj instances for calculations

In [ ]:
sy = Qobj([[0, -1j], [1j, 0]])  # the sigma-y Pauli operator

sy

In [ ]:
sz = Qobj([[1, 0], [0, -1]])  # the sigma-z Pauli operator

sz

In [ ]:
#some arithmetic with quantum objects

H = 1.0 * sz + 0.1 * sy

print("Qubit Hamiltonian = \n")
H

In [ ]:
# The hermitian conjugate
sy.dag()

In [ ]:
# The trace
H.tr()

In [ ]:
# Eigen energies
H.eigenenergies()

## States and operators

### State vectors

In [ ]:
# Fundamental basis states (Fock states of oscillator modes)

N = 2  # number of states in the Hilbert space
n = 1  # the state that will be occupied

basis(N, n)  # equivalent to fock(N, n)

In [ ]:
fock(4, 2)  # another example

In [ ]:
# a coherent state
coherent(N=10, alpha=1.0)

### 我们可以可视化一下Fock态还有coherent 态

In [ ]:
# 创建 Fock 态：|3⟩ 表示 3 个光子
n = 3
alpha = 2.0

fock_state = fock(10, n)
coherent_state = coherent(10, alpha)

plt.figure()
plot_fock_distribution(fock_state)
plt.title(f"Fock state |{n}>")

plt.figure()
plot_fock_distribution(coherent_state)
plt.title(f"Coherent state |{alpha}>")

plt.show()


In [ ]:
# 系统参数
N = 20         # 希尔伯特空间维度（截断）
n = 3          # Fock 态 |n⟩
alpha = 2.0    # 相干态幅度

# 创建量子态
fock_state = fock(N, n)
coh_state = coherent(N, alpha)

# 相空间网格
x = np.linspace(-5, 5, 200)
p = np.linspace(-5, 5, 200)
X, P = np.meshgrid(x, p)

# 计算 Wigner 分布
W_fock = wigner(fock_state, x, p)
W_coh = wigner(coh_state, x, p)

# 绘图
fig, axs = plt.subplots(1, 2, figsize=(12, 5))

# Fock 态图像
cf1 = axs[0].contourf(X, P, W_fock, 100, cmap='RdBu_r')
axs[0].set_title(f"Wigner Function: Fock state |{n}>")
axs[0].set_xlabel("x")
axs[0].set_ylabel("p")
fig.colorbar(cf1, ax=axs[0])

# Coherent 态图像
cf2 = axs[1].contourf(X, P, W_coh, 100, cmap='RdBu_r')
axs[1].set_title(f"Wigner Function: Coherent state |{alpha}>")
axs[1].set_xlabel("x")
axs[1].set_ylabel("p")
fig.colorbar(cf2, ax=axs[1])

plt.tight_layout()
plt.show()

In [ ]:
# 系统参数
N = 30
alpha = 2.0
omega = 1.0
tlist = np.linspace(0, 2 * np.pi, 60)

# 初始态和哈密顿量
psi0 = coherent(N, alpha)
H = omega * create(N) * destroy(N)
result = mesolve(H, psi0, tlist, [], [])

# 相空间网格
x = np.linspace(-5, 5, 200)
p = np.linspace(-5, 5, 200)
X, P = np.meshgrid(x, p)

# 绘图准备
fig, ax = plt.subplots(figsize=(6, 5))

def update(frame):
    ax.clear()
    w = wigner(result.states[frame], x, p)
    img = ax.contourf(X, P, w, 100, cmap="RdBu_r")
    ax.set_title(f"Wigner t = {tlist[frame]:.2f}")
    ax.set_xlabel("x")
    ax.set_ylabel("p")
    return img

ani = animation.FuncAnimation(fig, update, frames=len(tlist), interval=100)

# ✅ 直接显示动画
HTML(ani.to_jshtml())

In [ ]:
# 非经典初始态---Fock态叠加态
# 参数设置
N = 20
omega = 1.0
tlist = np.linspace(0, 2 * np.pi, 60)

# 创建叠加态：(|0⟩ + |3⟩)/√2
psi = (fock(N, 0) + fock(N, 3)).unit()

# 哈密顿量：谐振子 H = ω * a†a
H = omega * create(N) * destroy(N)

# 演化
result = mesolve(H, psi, tlist, [], [])

# 相空间网格
x = np.linspace(-5, 5, 200)
p = np.linspace(-5, 5, 200)
X, P = np.meshgrid(x, p)

# 动图绘制
fig, ax = plt.subplots(figsize=(6, 5))

def update(frame):
    ax.clear()
    w = wigner(result.states[frame], x, p)
    img = ax.contourf(X, P, w, 100, cmap="RdBu_r")
    ax.set_title(f"Wigner of (|0⟩+|3⟩)/√2  t = {tlist[frame]:.2f}")
    ax.set_xlabel("x")
    ax.set_ylabel("p")
    return img

ani = animation.FuncAnimation(fig, update, frames=len(tlist), interval=100)

# ✅ 显示动图
HTML(ani.to_jshtml())

In [ ]:
# 压缩态squeezed state
# 参数
N = 30
r = 0.7               # squeeze 参数，决定压缩程度
omega = 1.0
tlist = np.linspace(0, 2*np.pi, 60)

# 初始态：压缩真空态
psi = squeeze(N, r) * basis(N, 0)

# 哈密顿量：谐振子 H = a†a
H = omega * create(N) * destroy(N)

# 时间演化
result = mesolve(H, psi, tlist, [], [])

# 相空间网格
x = np.linspace(-5, 5, 200)
p = np.linspace(-5, 5, 200)
X, P = np.meshgrid(x, p)

# 动图初始化
fig, ax = plt.subplots(figsize=(6, 5))

def update(frame):
    ax.clear()
    w = wigner(result.states[frame], x, p)
    img = ax.contourf(X, P, w, 100, cmap="RdBu_r")
    ax.set_title(f"Squeezed state t = {tlist[frame]:.2f}")
    ax.set_xlabel("x")
    ax.set_ylabel("p")
    return img

ani = animation.FuncAnimation(fig, update, frames=len(tlist), interval=100)

# ✅ 在 Jupyter 中显示动图
HTML(ani.to_jshtml())

In [ ]:
# 压缩相干态（Squeezed Coherent State）
# 参数
N = 30
r = 0.7          # squeeze 参数
alpha = 2.0      # coherent 位移幅度
omega = 1.0
tlist = np.linspace(0, 2*np.pi, 60)

# 构造初始态：压缩+位移作用在真空态上
vacuum = basis(N, 0)
S = squeeze(N, r)
D = displace(N, alpha)
psi = D * S * vacuum

# 哈密顿量：H = a†a
H = omega * create(N) * destroy(N)

# 时间演化
result = mesolve(H, psi, tlist, [], [])

# 相空间网格
x = np.linspace(-5, 5, 200)
p = np.linspace(-5, 5, 200)
X, P = np.meshgrid(x, p)

# 动图初始化
fig, ax = plt.subplots(figsize=(6, 5))

def update(frame):
    ax.clear()
    w = wigner(result.states[frame], x, p)
    img = ax.contourf(X, P, w, 100, cmap="RdBu_r")
    ax.set_title(f"Squeezed Coherent t = {tlist[frame]:.2f}")
    ax.set_xlabel("x")
    ax.set_ylabel("p")
    return img

ani = animation.FuncAnimation(fig, update, frames=len(tlist), interval=100)

# ✅ 显示动画
HTML(ani.to_jshtml())


In [ ]:
# 模拟压缩相干态在环境中退相干（光子损失）
# 参数
N = 30
r = 0.7           # squeeze 强度
alpha = 2.0       # coherent 位移
omega = 1.0
kappa = 0.3       # photon loss rate
tlist = np.linspace(0, 8, 80)  # 更长时间看衰减过程

# 初始态：压缩 + 位移
vacuum = basis(N, 0)
S = squeeze(N, r)
D = displace(N, alpha)
psi = D * S * vacuum

# 哈密顿量
a = destroy(N)
H = omega * a.dag() * a

# 崩塌算符：光子损耗
c_ops = [np.sqrt(kappa) * a]

# Lindblad 演化
result = mesolve(H, psi, tlist, c_ops, [])

# 相空间网格
x = np.linspace(-5, 5, 200)
p = np.linspace(-5, 5, 200)
X, P = np.meshgrid(x, p)

# 动图绘制
fig, ax = plt.subplots(figsize=(6, 5))

def update(frame):
    ax.clear()
    w = wigner(result.states[frame], x, p)
    img = ax.contourf(X, P, w, 100, cmap="RdBu_r")
    ax.set_title(f"Decay | Squeezed Coherent  t = {tlist[frame]:.2f}")
    ax.set_xlabel("x")
    ax.set_ylabel("p")
    return img

ani = animation.FuncAnimation(fig, update, frames=len(tlist), interval=100)
HTML(ani.to_jshtml())

In [ ]:
# 参数
N = 30
alpha = 2.0
r = 0.7
omega = 1.0
kappa = 0.3
n_th = 0.5         # 热环境平均光子数
tlist = np.linspace(0, 8, 80)

# 初始态：压缩相干
vacuum = basis(N, 0)
S = squeeze(N, r)
D = displace(N, alpha)
psi = D * S * vacuum

# 算符
a = destroy(N)
H = omega * a.dag() * a

# 崩塌算符（热耗散）
c_ops = [
    np.sqrt(kappa * (1 + n_th)) * a,     # 光子发射
    np.sqrt(kappa * n_th) * a.dag()      # 热激发吸收
]

# Lindblad 主方程演化
result = mesolve(H, psi, tlist, c_ops, [])

# 相空间网格
x = np.linspace(-5, 5, 200)
p = np.linspace(-5, 5, 200)
X, P = np.meshgrid(x, p)

# 动画
fig, ax = plt.subplots(figsize=(6, 5))

def update(frame):
    ax.clear()
    w = wigner(result.states[frame], x, p)
    img = ax.contourf(X, P, w, 100, cmap="RdBu_r")
    ax.set_title(f"🔥 Squeezed Coherent + Thermal n_th={n_th}, t={tlist[frame]:.2f}")
    ax.set_xlabel("x")
    ax.set_ylabel("p")
    return img

ani = animation.FuncAnimation(fig, update, frames=len(tlist), interval=100)
HTML(ani.to_jshtml())


### Density matrices

In [ ]:
# a fock state as density matrix
fock_dm(5, 2)  # 5维Hilbert空间中的 |2>


In [ ]:
fock_state = fock(5, 2)  # 5维Hilbert空间中的 |2>
print(fock_state)
print(fock_state.full())

In [ ]:
# coherent state as density matrix
coherent_dm(N=8, alpha=1.0)

In [ ]:
coh_state = coherent(8, alpha=1.0)  # 截断在10维的相干态
print(coh_state.full())

In [ ]:
rho = ket2dm(coh_state)
print(rho.full())

In [ ]:
psi = coherent(15, alpha=2)
rho = ket2dm(psi)
plt.imshow(np.abs(rho.full()), cmap='viridis')
plt.title("Coherent State Density Matrix Magnitude")
plt.colorbar()
plt.show()

In [ ]:
# thermal state
n = 1  # average number of thermal photons
thermal_dm(8, n)

### Operators

#### Qubit (two-level system) operators

In [ ]:
# Pauli sigma x
sigmax()

In [ ]:
# Pauli sigma y
sigmay()

In [ ]:
# Pauli sigma z
sigmaz()

#### Harmonic oscillator operators

In [ ]:
#  annihilation operator

destroy(N=8)  # N = number of fock states included in the Hilbert space

In [ ]:
# creation operator

create(N=8)  # equivalent to destroy(5).dag()

In [ ]:
# the position operator is easily constructed from the annihilation operator
a = destroy(8)

x = a + a.dag()

x

In [ ]:
N = 10  # 截断的Fock空间维度

a = destroy(N)        # 湮灭算符 a
adag = create(N)      # 产生算符 a†
n_op = num(N)         # 数算符 a†a
x = (a + adag) / np.sqrt(2)  # 位置算符
p = 1j * (adag - a) / np.sqrt(2)  # 动量算符

print(a.full())       # 打印稠密形式

In [ ]:

coh = coherent(N, alpha=1.0)   # 相干态
rho = ket2dm(coh)
expect_vals = expect(n_op, rho)
print("平均光子数：", expect_vals)

#### Using Qobj instances we can check some well known commutation relations:

In [ ]:
def commutator(op1, op2):
    return op1 * op2 - op2 * op1

In [ ]:
a = destroy(5)

commutator(a, a.dag())

In [ ]:
x = (a + a.dag()) / np.sqrt(2)
p = -1j * (a - a.dag()) / np.sqrt(2)

In [ ]:
commutator(x, p)

In [ ]:
commutator(sigmax(), sigmay()) - 2j * sigmaz()

In [ ]:
-1j * sigmax() * sigmay() * sigmaz()


In [ ]:
sigmax() ** 2 == sigmay() ** 2 == sigmaz() ** 2 == qeye(2)

## Composite systems

In [ ]:
sz1 = tensor(sigmaz(), qeye(2))

sz1

In [ ]:
psi1 = tensor(basis(N, 1), basis(N, 0))  # excited first qubit
psi2 = tensor(basis(N, 0), basis(N, 1))  # excited second qubit

In [ ]:
# this should not be true,
# because sz1 should flip the sign of the excited state of psi1
sz1 * psi1 == psi1

In [ ]:
# this should be true, because sz1 should leave psi2 unaffected
sz1 * psi2 == psi2

In [ ]:
sz2 = tensor(qeye(2), sigmaz())

sz2

In [ ]:
tensor(sigmax(), sigmax())

In [ ]:
epsilon = [1.0, 1.0]
g = 0.1

sz1 = tensor(sigmaz(), qeye(2))
sz2 = tensor(qeye(2), sigmaz())

H = epsilon[0] * sz1 + epsilon[1] * sz2 + g * tensor(sigmax(), sigmax())

H

In [ ]:
wc = 1.0  # cavity frequency
wa = 1.0  # qubit/atom frenqency
g = 0.1  # coupling strength

# cavity mode operator
a = tensor(destroy(5), qeye(2))

# qubit/atom operators
sz = tensor(qeye(5), sigmaz())  # sigma-z operator
sm = tensor(qeye(5), destroy(2))  # sigma-minus operator

# the Jaynes-Cumming Hamiltonian
H = wc * a.dag() * a - 0.5 * wa * sz + g * (a * sm.dag() + a.dag() * sm)

H

In [ ]:
a = tensor(destroy(3), qeye(2))
sp = tensor(qeye(3), create(2))

a * sp

In [ ]:
tensor(destroy(3), create(2))

## Unitary dynamics

In [ ]:
# Hamiltonian
H = sigmax()

# initial state
psi0 = basis(2, 0)

# list of times for which the solver should store the state vector
tlist = np.linspace(0, 10, 100)

result = mesolve(H, psi0, tlist, [], [])

In [ ]:
result

In [ ]:
len(result.states)

In [ ]:
result.states[-1]  # the finial state

In [ ]:
fig, ani = anim_matrix_histogram(result, limits=[0, 1],
                                 bar_style='abs', color_style='phase')
# close an auto-generated plot and animation
plt.close()
ani

## Expectation values 

In [ ]:
expect(sigmaz(), result.states[-1])

In [ ]:
expect(sigmaz(), result.states)

In [ ]:
fig, axes = plt.subplots(1, 1)

axes.plot(tlist, expect(sigmaz(), result.states))

axes.set_xlabel(r"$t$", fontsize=20)
axes.set_ylabel(r"$\left<\sigma_z\right>$", fontsize=20);

In [ ]:
result = mesolve(H, psi0, tlist, [], [sigmax(), sigmay(), sigmaz()])

In [ ]:
fig, axes = plt.subplots(1, 1)

axes.plot(tlist, result.expect[2], label=r"$\left<\sigma_z\right>$")
axes.plot(tlist, result.expect[1], label=r"$\left<\sigma_y\right>$")
axes.plot(tlist, result.expect[0], label=r"$\left<\sigma_x\right>$")

axes.set_xlabel(r"$t$", fontsize=20)
axes.legend(loc=2);

In [ ]:
result.expect[0]

## Dissipative dynamics

In [ ]:
w = 1.0  # oscillator frequency
kappa = 0.1  # relaxation rate
a = destroy(10)  # oscillator annihilation operator
rho0 = fock_dm(10, 5)  # initial state, fock state with 5 photons
H = w * a.dag() * a  # Hamiltonian

# A list of collapse operators
c_ops = [np.sqrt(kappa) * a]

In [ ]:
tlist = np.linspace(0, 50, 100)

# request that the solver return the expectation value
# of the photon number state operator a.dag() * a
result = mesolve(H, rho0, tlist, c_ops, [a.dag() * a],
                 options={"store_states": True})

In [ ]:
fig, axes = plt.subplots(1, 1)
axes.plot(tlist, result.expect[0])
axes.set_xlabel(r"$t$", fontsize=20)
axes.set_ylabel(r"Photon number", fontsize=16);

In [ ]:
fig, ani = anim_fock_distribution(result)
# close an auto-generated plot and animation
plt.close()
ani